In [ ]:
import cv2
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torchvision import tv_tensors
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import (
    FetalBrainPlanesDataset,
    HeadSegmentationDataset,
    USVideosFrameDataset,
    USVideosSsimFrameDataset,
)
from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_numpy_images,
    show_pytorch_images,
)
from src.models.head_segmentation_module import HeadSegmentationLitModule

data_dir = root / "data"
dataset_name = "FETAL_HEAD_SEGMENTATION"
dataset_dir = data_dir / dataset_name

In [ ]:
%matplotlib ipympl

import matplotlib

matplotlib.use("ipympl")

# Visualize Data

In [ ]:
data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
predictions_df = pd.read_csv(dataset_dir / "prediction.csv")
len(predictions_df)

## Unidentified brain images

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 0]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # numbers
    # Ignore Brain images
    # ommited for now
]

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 0:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            images.append((image, f"{index}: {brain_plane} ({misidentified_row["Count"]})"))
            # print(f"{index:<6}: {image_name:<30} {brain_plane:<20} {misidentified_row["Count"]}")

print(len(images))
if images:
    show_pytorch_images(images[:25], gray=False).show()

## Not a brain images identify as brain image

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 1]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # Ignore Brain images
    # ommited for now
]

limit = 16

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 1:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            # images.append((image, f"{index}: {plane} ({misidentified_row["Count"]})"))
            images.append((image, f"{index}: ({misidentified_row["Count"]})"))
            # print(f"{index}, ")
            print(f"{index:<6}: {image_name:<30} {plane:<18} {misidentified_row["Count"]}")

            if len(images) >= limit:
                break
        if len(images) >= limit:
            break
    if len(images) >= limit:
        break

print(len(images))
if images:
    show_pytorch_images(images[:limit], gray=False).show()

In [ ]:
def display_image(img, ax, title):
    img = img.detach()
    img = TF.to_pil_image(img)
    img = TF.to_grayscale(img)
    ax.imshow(np.asarray(img), cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def overlap_mask(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def display_image_and_mask(image, mask):
    fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(15, 4))
    mask = mask.float()
    display_image(image, axes[0, 0], "Image")
    display_image(mask, axes[0, 1], "Mask")
    overlap_mask(image, mask, axes[0, 2], "Overlayed Mask")

    plt.tight_layout()
    plt.show()

    return


input_size = (165, 240)
transform = T.Compose(
    [
        T.Grayscale(),
        # RandomPercentCrop(max_percent=20),
        T.Resize(input_size, interpolation=InterpolationMode.NEAREST),
        # T.AutoAugment(T.AutoAugmentPolicy.IMAGENET),
        # T.RandAugment(magnitude=11),
        # T.TrivialAugmentWide(),
        # T.AugMix(),
        # T.RandomHorizontalFlip(p=0.5),
        # T.RandomVerticalFlip(p=0.5),
        # T.RandomAffine(degrees=20, translate=(0.1, 0.1), scale=(1.0, 1.2)),
        T.ToDtype(dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True),
        # T.Normalize(mean=[0.17], std=[0.19]),  # FetalBrain
        # T.Normalize(mean=[0.449], std=[0.226]),  # ImageNet
    ]
)

dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=None,
)

print(len(dataset))
image, mask, label = dataset[2013]
print("shape")
print(image.shape)
print(mask.shape)
print(label.shape)
print("values")
print(image.min(), image.max())
print(torch.unique(mask))
print(label)
print("")

dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=transform,
)

image_t, mask_t, label_t = dataset[2046]
print("shape")
print(image_t.shape)
print(mask_t.shape)
print(label_t.shape)
print("values")
print(image_t.min(), image_t.max())
print(torch.unique(mask_t))
print(label_t)
print("types")
print(mask_t.dtype)
print(label_t.dtype)
print("")

image_t2, mask_t2, label_t2 = dataset[6552]
print("shape")
print(image_t2.shape)
print(mask_t2.shape)
print(label_t2.shape)
print("values")
print(image_t2.min(), image_t2.max())
print(torch.unique(mask_t2))
print(label_t2)
print("types")
print(mask_t2.dtype)
print(label_t2.dtype)


display_image_and_mask(image, mask)
display_image_and_mask(image_t, mask_t)
display_image_and_mask(image_t2, mask_t2)
# show_pytorch_images([image, mask, image_t, mask_t]).show()

In [ ]:
ds = dataset.labels
ds = ds[ds["Patient_num"].notna()]
ds = ds[ds["Brain_plane"] == 1]
ds

In [ ]:
dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=T.Compose(
        [
            T.Grayscale(),
            PadToAspectRatio(
                (192, 256),
                fill={tv_tensors.Image: 0, tv_tensors.Mask: 0, "others": None},
            ),
            Resize(
                (192, 256),
                interpolation={
                    tv_tensors.Image: T.InterpolationMode.BILINEAR,
                    tv_tensors.Mask: T.InterpolationMode.NEAREST_EXACT,
                    "others": None,
                },
                # interpolation={tv_tensors.Image: "bilinear", tv_tensors.Mask: "nearest-exact", "others": None},
            ),
            T.ToDtype(
                dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True
            ),
        ]
    ),
)

image = dataset[0, 0]
image, mask, label = dataset[0]
print("shape")
print(image.shape)
print(mask.shape)
print(label.shape)
print("")
display_image_and_mask(image, mask)

In [ ]:
videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True),
    ),
)

fig, ax = plt.subplots()

video = videos[1]
frame = video[0]
frame = frame.detach()
frame = TF.to_pil_image(frame)
frame = TF.to_grayscale(frame)
ax.imshow(np.asarray(frame), cmap="gray")
ax.axis("off")

frames = []
for idx in tqdm(range(len(video))):
    frame = video[idx]
    frame = frame.detach()
    frame = TF.to_pil_image(frame)
    frame = TF.to_grayscale(frame)
    frame = ax.imshow(np.asarray(frame), cmap="gray", animated=True)
    frames.append([frame])

ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
plt.show()

In [ ]:
def display_image_animation(image, ax, title):
    image = image.detach()
    image = TF.to_pil_image(image)
    image = TF.to_grayscale(image)
    image = np.asarray(image)

    if not ax.images:
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, cmap="gray", animated=True)


def display_overlap_mask_animation(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    if not ax.images:
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, animated=True)


checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-25_13-36-03"
checkpoint = sorted(checkpoint_file.glob("checkpoints/epoch_*.ckpt"))[-1]
model = HeadSegmentationLitModule.load_from_checkpoint(checkpoint_path=str(checkpoint))
# disable randomness, dropout, etc...
model.eval()
model.to("cpu")

videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

# fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(10, 4))

# video = videos[1]
# frames = []
# for idx in tqdm(range(len(video))):
#     frame = video[idx]
#     with torch.no_grad():
#         logits = model(frame.unsqueeze(0)).squeeze(0)
#         prediction = (torch.sigmoid(logits) > 0.5).float()

#     im1 = display_image_animation(frame, axes[0, 0], "Image")
#     im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
#     im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")
#     frames.append([im1, im2, im3])

# ani = animation.ArtistAnimation(fig, frames, interval=100, blit=True, repeat_delay=1000)
# plt.tight_layout()
# plt.show()

In [ ]:
def predict_head_mask(model: HeadSegmentationLitModule, image: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Binary mask [1, H, W] and frame label (same rules as HeadSegmentationLitModule.calculate_prediction)."""
    logits = model(image.unsqueeze(0))
    binary_mask, prediction_labels = model.calculate_hard_prediction(logits)
    return binary_mask.squeeze(0), int(prediction_labels.squeeze(0).item())


def find_angle(mask):
    mask = mask.squeeze(0)

    # Get the indices of the non-zero elements
    coords = torch.nonzero(mask, as_tuple=False).float()
    # Center the data by subtracting the mean
    mean = torch.mean(coords, dim=0)
    centered_coords = coords - mean

    # Calculate the covariance matrix
    covariance_matrix = torch.matmul(centered_coords.T, centered_coords)
    # Perform eigenvalue decomposition to find eigenvectors (principal components)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance_matrix)
    # The eigenvector corresponding to the largest eigenvalue (the first principal component)
    principal_axis = eigenvectors[:, 1]
    # Calculate the angle of the principal axis
    angle = torch.atan2(principal_axis[0], principal_axis[1]) * 180 / torch.pi

    return angle.round().int()


video = videos[1]
frame = video[0]
# frame = TF.rotate(frame, angle=120)
with torch.no_grad():
    prediction, _ = predict_head_mask(model, frame)

for base_angle in range(0, 360, 30):
    prediction2 = TF.rotate(prediction, angle=base_angle)
    angle = find_angle(prediction2)
    print(f"Base angle: {base_angle}, angle: {angle}, result: {base_angle + angle}")
# break

In [ ]:
def crop(image, x1, y1, x2, y2, pad=10):
    pad_x = int((x2 - x1) * (pad / 100.0))
    left_pad = pad_x // 2
    right_pad = pad_x - left_pad

    pad_y = int((y2 - y1) * (pad / 100.0))
    top_pad = pad_y // 2
    bottom_pad = pad_y - top_pad

    x1 = max(x1 - left_pad, 0)
    y1 = max(y1 - top_pad, 0)
    x2 = min(x2 + right_pad, image.shape[2])
    y2 = min(y2 + bottom_pad, image.shape[1])
    return TF.crop(image, y1, x1, y2 - y1, x2 - x1)


def pad_central(image, height, width):
    original_height, original_width = image.shape[1:]

    padding_needed = height - original_height
    top_pad = padding_needed // 2
    bottom_pad = padding_needed - top_pad

    padding_needed = width - original_width
    left_pad = padding_needed // 2
    right_pad = padding_needed - left_pad

    return TF.pad(image, padding=[left_pad, top_pad, right_pad, bottom_pad], fill=0)


class StabilizeVideoEma:
    def __init__(self, alpha: float = 0.5):
        self.alpha = alpha
        self.args = None
        self.int_type = False

    def stabilize(self, *args):
        if self.args is None:
            self.args = self._extract_item_from_tensor(args)

            for arg in self.args:
                if isinstance(arg, int):
                    self.int_type = True
        else:
            for i in range(len(self.args)):
                self.args[i] = self.alpha * args[i] + (1 - self.alpha) * self.args[i]

        rs = self._extract_item_from_tensor(self.args)
        rs = [int(round(arg)) if self.int_type else arg for arg in rs]

        if len(rs) == 1:
            return rs[0]
        else:
            return rs

    def _extract_item_from_tensor(self, args):
        return [args[i].item() if isinstance(args[i], torch.Tensor) else args[i] for i in range(len(args))]


videos = USVideosFrameDataset(
    data_dir=data_dir,
    train=False,
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

fig, axes = plt.subplots(ncols=3, nrows=3, squeeze=False, figsize=(10, 8))
video = videos[0]
frames = []

image_size = (165, 240)
interpolation = T.InterpolationMode.NEAREST

frame_crop_prev = None
frame_rotate_crop_prev = None

alpha = 0.5
ema_stabilizer_crop = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop = StabilizeVideoEma(alpha=alpha)

alpha = 0.1
ema_stabilizer_crop_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop_2 = StabilizeVideoEma(alpha=alpha)

pad = 5
image_pad_central = (700, 900)
# image_size_crop=(175, 225)
image_size_crop = (88, 112)

for idx in tqdm(range(len(video))):
    frame_base = video[idx]
    frame = TF.resize(frame_base, size=image_size, interpolation=interpolation)
    with torch.no_grad():
        prediction, _ = predict_head_mask(model, frame)
        prediction_base = TF.resize(prediction, size=frame_base.shape[1:], interpolation=interpolation)

    # --------------------------------------------
    im1 = display_image_animation(frame, axes[0, 0], "Image")
    im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
    im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im4 = display_image_animation(frame_2, axes[1, 0], "Crop")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im5 = display_image_animation(frame_2, axes[1, 1], "ema-stabilized")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im6 = display_image_animation(frame_2, axes[1, 2], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im7 = display_image_animation(frame_2, axes[2, 0], "Rotate-crop")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im8 = display_image_animation(frame_2, axes[2, 1], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate_2.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im9 = display_image_animation(frame_2, axes[2, 2], "ema-stabilized")

    # --------------------------------------------
    frames.append([im1, im2, im3, im4, im5, im6, im7, im8, im9])

ani = animation.ArtistAnimation(fig, frames, interval=10, blit=True, repeat_delay=1000)
plt.tight_layout()
plt.show()

In [ ]:
# First, close any open figures
plt.close("all")

# Then, restart the ipympl backend by rerunning the magic command
%matplotlib ipympl